In [ ]:
### Build a basic chatbot with GRAPH API


In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langchain_core.messages import HumanMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langchain_groq import ChatGroq
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver
from IPython.display import Image, display


In [ ]:
class State(TypedDict):
    messages: Annotated[list, add_messages]


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
llm = ChatGroq(model='openai/gpt-oss-20b')


In [ ]:
def chatbot(state: State):
    return {"messages": [llm.invoke(state["messages"])]}


In [ ]:
graph_builder = StateGraph(State)
graph_builder.add_node('llamabot', chatbot)
graph_builder.add_edge(START, 'llamabot')
graph_builder.add_edge('llamabot', END)
graph = graph_builder.compile()


In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass


In [ ]:
graph.invoke({'messages': [HumanMessage(content='Hi')]})


In [ ]:
for event in graph.stream({'messages': [HumanMessage(content='Hi How are you?')]}):
    for value in event.values():
        print(value['messages'][-1].content)


In [ ]:
tool = TavilySearch(max_results=2)


In [ ]:
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


In [ ]:
tools = [tool, multiply]


In [ ]:
llm_with_tool = llm.bind_tools(tools)
llm_with_tool


In [ ]:
def tool_calling_llm(state: State):
    return {"messages": [llm_with_tool.invoke(state['messages'])]}


In [ ]:
builder = StateGraph(State)
builder.add_node('tool_calling_llm', tool_calling_llm)
builder.add_node('tools', ToolNode(tools))

builder.add_edge(START, 'tool_calling_llm')
builder.add_conditional_edges('tool_calling_llm', tools_condition)
builder.add_edge('tools', END)

graph = builder.compile()


In [ ]:
try:
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception:
    pass


In [ ]:
response = graph.invoke({'messages': [HumanMessage(content='What is the recent AI news')]})


In [ ]:
response['messages'][-1].content


In [ ]:
for m in response['messages']:
    print(m.content)


In [ ]:
response = graph.invoke({'messages': [HumanMessage(content='What is 5 multiplied by 2')]})
for m in response['messages']:
    print(m.content)


In [ ]:
memory = MemorySaver()

builder = StateGraph(State)
builder.add_node('tool_calling_llm', tool_calling_llm)
builder.add_node('tools', ToolNode(tools))

builder.add_edge(START, 'tool_calling_llm')
builder.add_conditional_edges('tool_calling_llm', tools_condition)
builder.add_edge('tools', END)

graph = builder.compile(checkpointer=memory)


In [ ]:
config = {'configurable': {'thread_id': '1'}}
response = graph.invoke({'messages': [HumanMessage(content='My name is ___')]}, config=config)
response


In [ ]:
response['messages'][-1].content


In [ ]:
response = graph.invoke({'messages': [HumanMessage(content='Hey whats my name')]}, config=config)
print(response['messages'][-1].content)
